# cctv_dfd smoke test (Colab T4 / Kaggle)

End-to-end check: install -> face crop 200 local samples -> train 1 epoch ->
evaluate over 5 degradation profiles × 2 enhancement modes -> explain one
sample -> verify forensic log.

**Runtime:** under 10 minutes on a free T4 instance.

## 1. Setup

In [ ]:
# If running on Colab, clone the repo first.
# from google.colab import drive  # optional
# drive.mount('/content/drive')
# !git clone <your-fork-url>  /content/Deepfake-detection
%cd /content/Deepfake-detection/cctv_dfd  # adjust path if needed
!pip install -q -e .
!pip install -q facenet-pytorch

In [ ]:
import torch, transformers, albumentations
print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('albumentations', albumentations.__version__)

## 2. Build a tiny local dataset from existing extracted frames

In [ ]:
!python -m cctv_dfd.cli face-extract \
  --input ../Deepfake_Detection_Project/Deepfake_Detection_Project/data \
  --output data/processed/local \
  --limit 200 --already-cropped

In [ ]:
!ls data/processed/local/Real | head
!ls data/processed/local/Fake | head
!echo '---' && ls data/processed/local/Real | wc -l && ls data/processed/local/Fake | wc -l

## 3. Train 1 epoch

In [ ]:
!python -m cctv_dfd.cli train \
  --config configs/dinov2s_mlp.yaml \
  --dataset local --epochs 1 --batch-size 32 --tag smoke

## 4. Stratified evaluation

In [ ]:
!python -m cctv_dfd.cli eval \
  --run-tag smoke --dataset local \
  --profiles all --enhance-modes none forensic

In [ ]:
import pathlib, json
p = pathlib.Path('results/metrics/smoke_eval.json')
data = json.loads(p.read_text())
for row in data['rows']:
    print(f"{row['dataset']:8s} {row['profile']:16s} {row['enhance']:9s} "
          f"auc={row['auc']:.3f} f1={row['f1']:.3f} acc={row['acc']:.3f} n={row['n']}")

## 5. Explain a single sample + forensic log

In [ ]:
from pathlib import Path
fake = next(iter(Path('data/processed/local/Fake').glob('*.jpg')))
print('target:', fake)
!python -m cctv_dfd.cli predict --run-tag smoke --image {fake} --enhance forensic

In [ ]:
!cat results/forensic.jsonl | tail -n 1 | python -m json.tool
!ls -la reports/ | head

## Pass criteria
- `results/checkpoints/smoke.pt` exists
- `results/metrics/smoke_eval.json` has 5 profiles × 2 enhance modes = 10 rows for the `local` dataset
- one HTML file appeared under `reports/`
- `results/forensic.jsonl` has at least one valid JSON line with a non-empty `input_sha256` and `model_sha256`

If all four hold, the pipeline is wired end-to-end. Next step: run
`01_train_dinov2s.ipynb` on a real FF++ subset.